In [ ]:
# Initial required imports
# This portion was developed through Google Colab

!pip install amrlib
!pip install -q amrlib
!python -m spacy download en_core_web_sm
!pip install unidecode penman torch torchvision torchaudio --quiet

import amrlib
import tarfile
import urllib.request
from pathlib import Path

amrlib_dir = Path(amrlib.__file__).resolve().parent
data_dir = amrlib_dir / "data"
data_dir.mkdir(exist_ok=True)

# Github link to the pretrained AMR graph model. It converts text to AMR graphs.
url = "https://github.com/bjascob/amrlib-models/releases/download/parse_xfm_bart_large-v0_1_0/model_parse_xfm_bart_large-v0_1_0.tar.gz"

tar_path = data_dir / "model_parse_xfm_bart_large-v0_1_0.tar.gz"
extracted_dir = data_dir / "model_parse_xfm_bart_large-v0_1_0"
target_dir = data_dir / "model_stog"

if not tar_path.exists():
    urllib.request.urlretrieve(url, tar_path)

if not extracted_dir.exists():
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=data_dir)

if not target_dir.exists():
    extracted_dir.rename(target_dir)

# print("installed parser model at:", target_dir)

# Import the appropriate files from Drive into Colab
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
INPUT_FILE = "/content/drive/My Drive/COMP459/outputs/all_data.jsonl"
!ls "{INPUT_FILE}"

# (More) imports
!pip install torch-geometric penman sentence-transformers networkx spacy
!python -m spacy download en_core_web_trf
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import penman
import numpy as np
import networkx as nx
import spacy
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
# print("device:", device)

# Step 1: load data

def load_records(path):
    """Open the input path file and return the records as a list."""
    with open(path) as f:
        return [json.loads(l) for l in f]

# Use input file
records = load_records(INPUT_FILE)
# print(f"loaded {len(records)} records")

# Step 2: node feature embedding
"""
Each AMR node has a concept label (such as "person" or "get-01").
In this step, we embed these concept strings thorugh sentence transformers to
turn them into 384 dimension vectors.

GNN: concept embedding (num_nodes × 384) + edge_index fed into GCNConv
"""
# embed strings using SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def get_concept_embedding(concept_str, embedder):
    """
    Removes the sense numbers from a concept, and then turns the word
    into a numeric vector (embedding).

    So, "get-01" would turn into an embedding of "get".

    """
    base = concept_str.split("-")[0]
    # returns a shape (384,)
    return embedder.encode(base, convert_to_tensor=True)


# Step 3: Build a document level NetworkX graph from a list of AMR strings
def build_doc_graph(amr_strings):
    """
    Build one document level graph from a list of sentence level AMR strings.

    Each AMR string represents one sentence. This function combines them into
    a single NetworkX directed graph so the entire document can be processed as
    one graph.

    The final graph contains:
    1. AMR semantic edges: These come from each AMR graph, such as :ARG0.
    2. Sentence flow edges: These edges connect the root of sentence 0 to the
    root of sentence 1, sentence 1 to 2, and so on, to preserve document order.
    3. Coreference edges: These eges connect nodes in different sentences if they
    have the same base concept, such as "person-01" and "person-02" both representing
    "person".

    The node naming convention is "{sentence index}::{AMR variable}". So, if
    sentence 0 has an AMR variable "b", the node ID is "0::b". We did this to ensure
    that collisions were avoided, since different sentence AMRs often reuse
    variable names. Two sentences might both use a variable called "p", so sentence
    index is added.
    """
    # create empty directed graph
    G = nx.DiGraph()
    # to preserve sentence order
    sentence_roots = []
    # to add coref edges between repeated concepts
    concept_to_nodes = defaultdict(list)

    for sent_idx, amr_str in enumerate(amr_strings):
        try:
            g = penman.decode(amr_str)
        except Exception:
            continue

        instances = g.instances()
        if not instances:
            continue

        sent_root = None

        # add nodes for this AMR
        for inst in instances:
            node_id = f"{sent_idx}::{inst.source}"
            base_concept = inst.target.split("-")[0]
            G.add_node(node_id,
                       concept=inst.target,
                       base_concept=base_concept,
                       sent_idx=sent_idx)
            concept_to_nodes[base_concept].append(node_id)
            if sent_root is None:
                sent_root = node_id

        if sent_root:
            sentence_roots.append(sent_root)

        # add semantic role edges
        for edge in g.edges():
            src_id = f"{sent_idx}::{edge.source}"
            tgt_id = f"{sent_idx}::{edge.target}"
            if G.has_node(src_id) and G.has_node(tgt_id):
                G.add_edge(src_id, tgt_id, etype="semantic", role=edge.role)
                G.add_edge(tgt_id, src_id, etype="semantic", role=edge.role + "_inv")
    # add flow (discourse) edges
    for i in range(len(sentence_roots) - 1):
        G.add_edge(sentence_roots[i], sentence_roots[i + 1],
                   etype="discourse", role="NEXT_SENT")

    # add coreference edges
    for base_concept, node_ids in concept_to_nodes.items():
        if len(node_ids) > 1:
            for i in range(len(node_ids)):
                for j in range(i + 1, len(node_ids)):
                    ni, nj = node_ids[i], node_ids[j]
                    si = G.nodes[ni].get("sent_idx", -1)
                    sj = G.nodes[nj].get("sent_idx", -1)
                    if si != sj:
                        G.add_edge(ni, nj, etype="coref", role="COREF")
                        G.add_edge(nj, ni, etype="coref", role="COREF")

    return G


def doc_graph_to_pyg(nx_graph, label, embedder):
    """
    Convert a NetworkX document-level graph into a PyG Data object.
    """
    nodes = list(nx_graph.nodes())
    if not nodes:
        return None

    node_to_idx = {n: i for i, n in enumerate(nodes)}

    x = torch.stack([
        get_concept_embedding(
            nx_graph.nodes[n].get("concept", n.split("::")[-1]),
            embedder
        )
        for n in nodes
    ])

    src_list, tgt_list = [], []
    for src, tgt in nx_graph.edges():
        if src in node_to_idx and tgt in node_to_idx:
            src_list.append(node_to_idx[src])
            tgt_list.append(node_to_idx[tgt])

    if src_list:
        edge_index = torch.tensor([src_list, tgt_list], dtype=torch.long)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)

    y = torch.tensor([label], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, y=y)


# Step 4: Train/val split
train_records, val_records = train_test_split(
    records, test_size=0.2, random_state=42
)
# print(f"train records: {len(train_records)} | val/test records: {len(val_records)}")

dataset = []
for r in train_records:
    normal_amrs = r["normal_amr"] if isinstance(r["normal_amr"], list) else [r["normal_amr"]]
    doc_G_normal = build_doc_graph(normal_amrs)
    pyg_normal = doc_graph_to_pyg(doc_G_normal, label=0, embedder=embedder)
    if pyg_normal is not None:
        dataset.append(pyg_normal)

    doc_G_easy = build_doc_graph(r["easy_amr"])
    pyg_easy = doc_graph_to_pyg(doc_G_easy, label=1, embedder=embedder)
    if pyg_easy is not None:
        dataset.append(pyg_easy)

# print(f"document-level graphs built: {len(dataset)}")
# print(f"  standard (0): {sum(1 for d in dataset if d.y.item() == 0)}")
# print(f"  easy read (1): {sum(1 for d in dataset if d.y.item() == 1)}")

train_data, val_data = train_test_split(dataset, test_size=0.2, random_state=42,
                                         stratify=[d.y.item() for d in dataset])
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=32, shuffle=False)

# Step 5: Define the GNN model

class EasyReadGNN(nn.Module):
    def __init__(self, in_channels=384, hidden_channels=128, dropout=0.4):
        super().__init__()

        # first graph convolution
        self.conv1 = GCNConv(in_channels, hidden_channels)

        # second graph convolution
        self.conv2 = GCNConv(hidden_channels, hidden_channels)

        # fully connected layer
        self.fc1  = nn.Linear(hidden_channels, 64)

        # output
        self.fc_out = nn.Linear(64, 1)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        """Completes a forward pass of the GNN."""
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.fc_out(x).squeeze(-1)


# Step 6: Training loop

def train(model, loader, optimizer, loss_fn, epoch, verbose=False):
    """One training epoch over the dataset of graphs."""
    model.train()
    total_loss = 0
    for i, batch in enumerate(loader):
        batch = batch.to(device)
        pred = model(batch.x, batch.edge_index, batch.batch)
        loss = loss_fn(pred, batch.y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        if verbose and (i % 20 == 0):
            print(f"  [epoch {epoch}: batch {i}/{len(loader)}] loss: {loss.item():.4f}")
    avg = total_loss / len(loader)
    print(f"epoch {epoch} | avg train loss: {avg:.4f}")
    return avg


# Step 7: Validation loop

def validate(model, loader, loss_fn):
    """Evaluate the given model on the validation dataset."""
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch.x, batch.edge_index, batch.batch)
            total_loss += loss_fn(pred, batch.y).item()
            binary = (torch.sigmoid(pred) > 0.5).long().cpu().tolist()
            all_preds.extend(binary)
            all_labels.extend(batch.y.long().cpu().tolist())
    avg = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds)
    # print(f"val loss: {avg:.4f} | acc: {acc:.3f} | F1: {f1:.3f}")
    return avg


# Step 8: Run training

model = EasyReadGNN(in_channels=384, hidden_channels=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

train_losses, val_losses = [], []
epochs = 30

# implement early stopping here
best_val = float("inf")
patience = 4
bad_epochs = 0

# training loop
for e in range(1, epochs + 1):
    train_loss = train(model, train_loader, optimizer, loss_fn, e, verbose=True)
    val_loss   = validate(model, val_loader, loss_fn)

    # store losses
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # if validation improves, save the model
    if val_loss < best_val:
        best_val = val_loss
        bad_epochs = 0
        torch.save(model.state_dict(), "best_gnn.pt")
    else:
        # if no improvement, stop the training before set number of epochs
        bad_epochs += 1
        if bad_epochs >= patience:
            print("Early stopping triggered")
            break

model.load_state_dict(torch.load("best_gnn.pt"))

# Plotting the model's performance
import matplotlib.pyplot as plt
plt.plot(train_losses, label="Train")
plt.plot(val_losses,   label="Val")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.title("GNN Loss")
plt.show()

# Step 9: Extract GNN embeddings and turn them into readable LLM prompts

"""
After training, the GNN produces a graph level embedding. However, we need to
convert it from a 128 dimension vector into useful guidance for the LLMs. In
order to interface with the LLM, we will do the following here:
1. get_graph_embedding() extracts the vector.
2. embedding_to_structural_stats() computes human readable graph features from the
original graph to reflect what the GNN has learned internally about easy read structure.
3. derive_structural_guidance() converts the features into a plain English block
to insert into the LLM prompt.
4. build_llm_prompt() assembles the full three-part prompt:
    Part 1 – Easy Read rules (hardcoded)
    Part 2 – GNN-derived structural guidance (from this function)
    Part 3 – Knowledge graph with ranked concepts (from Step 10)
"""

def get_graph_embedding(model, data_obj):
    """
    Retrieves the graph level embedding for a single document graph.
    This runs the GNN forward without the final prediction layer and returns
    the pooled node representation as a vector.
    """
    model.eval()
    with torch.no_grad():
        data_obj = data_obj.to(device)
        x = F.relu(model.conv1(data_obj.x, data_obj.edge_index))
        x = F.relu(model.conv2(data_obj.x, data_obj.edge_index))
        embedding = global_mean_pool(x, torch.zeros(x.size(0), dtype=torch.long, device=device))
        return embedding.cpu().numpy()


def embedding_to_structural_stats(nx_doc_graph):
    """
    Derives interpretable statistics from the document level NetworkX graph.
    These stats reflect what's been learned by the GNN but in a way that
    is easily understandable to an LLM.

    We will use the AMR graph built by build_doc_graph() here, not from
    extract_knowledge_graph().

    Returns a dictionary with the following:
      num_nodes: total concept nodes across all sentences
      num_sentences: number of sentence level AMR graphs merged
      avg_nodes_per_sent: average nodes per sentence
      graph_density: edge density (0 is sparse, 1 is fully connected)
      max_path_length: longest shortest path
      num_coref_edges: numnber of cross-sentence coreference edges
      top_concepts: top 5 highest degree concepts
    """
    G = nx_doc_graph
    if G.number_of_nodes() == 0:
        return {
            "num_nodes": 0, "num_sentences": 1, "avg_nodes_per_sent": 0.0,
            "graph_density": 0.0, "max_path_length": 0,
            "num_coref_edges": 0, "top_concepts": [],
        }

    num_nodes = G.number_of_nodes()

    # sentence count
    sent_indices = {
        data["sent_idx"]
        for _, data in G.nodes(data=True)
        if "sent_idx" in data
    }
    if sent_indices:
        num_sentences = len(sent_indices)
    else:
        discourse_edges = sum(
            1 for _, _, d in G.edges(data=True)
            if d.get("role") == "NEXT_SENT"
        )
        num_sentences = max(discourse_edges + 1, 1)

    avg_nodes_per_sent = round(num_nodes / num_sentences, 1)
    density = round(nx.density(G), 3)

    # max shortest path
    G_und = G.to_undirected()
    try:
        largest_cc = max(nx.connected_components(G_und), key=len)
        sub = G_und.subgraph(largest_cc)
        max_path = nx.diameter(sub)
    except Exception:
        max_path = 0

    num_coref = sum(
        1 for _, _, d in G.edges(data=True) if d.get("etype") == "coref"
    )

    degree_map = {
        G.nodes[n].get("base_concept", n): G.degree(n)
        for n in G.nodes()
    }
    top_concepts = sorted(degree_map.items(), key=lambda x: x[1], reverse=True)[:5]

    return {
        "num_nodes": num_nodes,
        "num_sentences": num_sentences,
        "avg_nodes_per_sent": avg_nodes_per_sent,
        "graph_density": density,
        "max_path_length": max_path,
        "num_coref_edges": num_coref,
        "top_concepts": top_concepts,
    }

# these are AMR semantic role placeholders
# when they appear in the "central concepts", the LLM may try to preserve the
# literal word "person" or "have" even if the original text doesn't use them
AMR_ABSTRACT_CONCEPTS = {
    "and", "or", "contrast", "have", "be", "do", "make", "get",
    "give", "say", "go", "know", "take", "see", "come", "want",
    "person", "thing", "place", "time", "way", "number", "name",
    "country", "organization", "before", "after", "cause",
    "multi-sentence", "relative-position", "they", "it", "she", "he",
    "we",
}


def derive_structural_guidance(stats, easy_read_baseline=None):
    """
    Converts graph level structural statistics into plain English simplification
    instructions.
    """
    if easy_read_baseline is None:
        easy_read_baseline = {
            "avg_nodes_per_sent": 5.5,
            "max_path_length": 4,
            "graph_density": 0.35,
        }

    lines = ["STRUCTURAL GUIDANCE (from Easy Read graph analysis):"]

    # complexity and split guidance
    current = stats["avg_nodes_per_sent"]
    target  = easy_read_baseline["avg_nodes_per_sent"]
    if current > target * 1.3:
        lines.append(
            f"- This document averages {current} concepts per sentence "
            f"(Easy Read target: ~{target}). "
            f"Split long sentences so each expresses only one idea."
        )
    else:
        lines.append(
            f"- Sentence complexity is close to Easy Read level "
            f"({current} concepts/sentence). Maintain this simplicity."
        )

    # depth and nesting guidance
    depth = stats["max_path_length"]
    target_depth = easy_read_baseline["max_path_length"]
    if depth > target_depth:
        lines.append(
            f"- The concept chain depth is {depth} steps "
            f"(Easy Read target: ≤{target_depth}). "
            f"Avoid nested clauses; flatten each idea to a direct statement."
        )
    else:
        lines.append(
            f"- Concept chain depth ({depth}) is within Easy Read range. "
            f"Keep sentences direct and avoid sub-clauses."
        )


    # density and connectivity guidance
    density = stats["graph_density"]
    if density < 0.15:
        lines.append(
            "- The document graph is sparse. Make logical connectives "
            "explicit in the output ('then', 'because', 'so', 'also')."
        )
    elif density > 0.5:
        lines.append(
            "- High concept overlap detected. Consolidate repeated ideas "
            "rather than restating them across sentences."
        )


    # coreference guidance
    coref = stats["num_coref_edges"]
    if coref == 0:
        lines.append(
            "- No cross-sentence entity links detected. "
            "Use pronoun references or topic sentences to link ideas "
            "('He', 'This', 'They') so the output reads as connected text."
        )
    else:
        lines.append(
            f"- {coref} cross-sentence coreference links detected. "
            f"Preserve these references to maintain topic coherence."
        )

    num_nodes = stats["num_nodes"]
    target_sents = max(2, round(num_nodes / 5.5))
    lines.append(
        f"- Aim for around {target_sents} short sentences, but do not pad the output. "
        f"If the meaning is complete in fewer sentences, stop there. "
        f"Never add information that is not in the original text."
    )

    # AMR uses generic placeholders like "person", "have", "and", etc. as
    # semantic role nodes. We'll filter these out because telling the LLM to
    # preserve these will produce strange and incorrect outputs.
    # For example, it might use "person" instead of "musician".
    if stats["top_concepts"]:
        surface_concepts = [
            c for c, _ in stats["top_concepts"]
            if c.lower() not in AMR_ABSTRACT_CONCEPTS
        ]
        if surface_concepts:
            vocab = ", ".join(surface_concepts)
            lines.append(
                f"- Central concepts in this document: {vocab}. "
                f"Use these terms consistently in the output."
            )

    return "\n".join(lines)

# Step 10: Knowledge graph extraction pipeline

def extract_knowledge_graph(text):
    """
    Builds a knowledge graph from the provided text.

    Given a document string, the function returns:
    1. G: NetworkX graph where nodes are tokens or named entities and edges
    come from the spaCy dependency links.
    2. centrality: dictionary of {concept: score} for the content words.
    3. ranked: list of concepts sorted from most to least central
    """
    nlp = spacy.load("en_core_web_trf")
    doc = nlp(text)

    G = nx.DiGraph()

    # build a map from token index to entity so multiple word entities are
    # treated as single nodes. ("New York" for example)
    token_to_ent = {}
    for ent in doc.ents:
        G.add_node(ent.text, type=ent.label_, is_ent=True, is_stop=False)
        for token in ent:
            token_to_ent[token.i] = ent.text

    # add every remaining token as a node
    CONTENT_POS = {"NOUN", "PROPN", "VERB", "NUM", "ADJ"}
    for token in doc:
        if token.i in token_to_ent:
            continue
        node_id = token.text
        if node_id not in G:
            is_content = (
                token.pos_ in CONTENT_POS
                and not token.is_stop
                and not token.is_punct
            )
            G.add_node(node_id,
                       type=token.pos_,
                       is_ent=False,
                       is_stop=(not is_content))

    # add dependency edges
    # each token points to its syntactic head
    # if a token belongs to an entity, use that instead
    for token in doc:
        src_id = token_to_ent.get(token.i, token.text)
        tgt_id = token_to_ent.get(token.head.i, token.head.text)
        if src_id != tgt_id and G.has_node(src_id) and G.has_node(tgt_id):
            G.add_edge(src_id, tgt_id, relation=token.dep_)

    # score nodes by degree centrality
    full_centrality = nx.degree_centrality(G)

    # keep only useful content concepts for the prompt
    # the graph might contain stopwords but this concept list should not
    centrality = {
        node: score
        for node, score in full_centrality.items()
        if not G.nodes[node].get("is_stop", True)
        and not G.nodes[node].get("type", "") in {"PUNCT", "SPACE"}
    }

    # sort concepts from most central to least central
    ranked = sorted(centrality.items(), key=lambda kv: kv[1], reverse=True)
    return G, centrality, ranked


# Step 11: LLM prompt builder
"""
Creates the three-part prompt using:
   Part 1 – hardcoded Easy Read rules: instructions about sentence length, clarity,
   plain language, etc.
   Part 2 – GNN-derived structural guidance: generated from the AMR document graph,
   it might tell the LLM to split sentences, add connectives, reduce nested clauses, etc.
   Part 3 – knowledge graph with ranked concepts: generated from the spaCy knowledge
   graph, it tells the LLM which important concepts should be preserved.
"""

def build_llm_prompt(original_text, nx_doc_graph, knowledge_graph_ranked):
    """
    Build an Easy Read simplification prompt for an LLM.

    original_text is a string of text that needs to be simplified.
    nx_doc_graph is the AMR based document graph.
    knowledge_graph_ranked is a ranked list of (concept, score) pairs.
    """
    stats = embedding_to_structural_stats(nx_doc_graph)
    structural_guidance = derive_structural_guidance(stats)

    top_concepts = knowledge_graph_ranked[:10]
    if top_concepts:
        top_score = top_concepts[0][1]
        threshold = top_score * 0.5

        must      = [(c, s) for c, s in top_concepts if s >= threshold]
        contextual = [(c, s) for c, s in top_concepts if s < threshold]

        must_lines = "\n".join(
            f"  {i+1}. {c}"
            for i, (c, _) in enumerate(must)
        )
        concept_block = f"KEY CONCEPTS — you MUST use these words in your output:\n{must_lines}"

        if contextual:
            ctx_lines = "\n".join(
                f"  {i+1}. {c}"
                for i, (c, _) in enumerate(contextual)
            )
            concept_block += f"\n\nSUPPORTING CONCEPTS — include these only if they fit naturally:\n{ctx_lines}"
    else:
        concept_block = "KEY CONCEPTS: (none extracted)"

    prompt = f"""You are simplifying a document into Easy Read format for people with intellectual disabilities.

RULES:
- Use short sentences (8–12 words each).
- Use active voice.
- Express only one idea per sentence.
- Use simple, everyday words.
- Do not use jargon or technical terms without explaining them.
- Only use information from the original text. Do not add explanations,
  motivations, or details that are not explicitly stated.

{concept_block}

{structural_guidance}

ORIGINAL TEXT:
\"\"\"{original_text}\"\"\"

SIMPLIFIED OUTPUT:"""

    return prompt

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

import os, json, time
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

def call_llm(client: OpenAI, prompt: str, model="gpt-4o-mini", max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(2 ** attempt)
    return None

def build_baseline_prompt(original_text):
    return f"""You are simplifying a document into Easy Read format for people with intellectual disabilities.

RULES:
- Use short sentences, about 8-12 words each.
- Use active voice.
- Express only one idea per sentence.
- Use simple, everyday words.
- Do not use jargon or technical terms without explaining them.
- Keep the important meaning of the original text.

ORIGINAL TEXT:
\"\"\"{original_text}\"\"\"

SIMPLIFIED OUTPUT:"""

def get_reference(r):
    rewritten = r.get("easy_read_rewritten")
    if rewritten:
        return " ".join(rewritten) if isinstance(rewritten, list) else rewritten
    easy = r.get("easy_text")
    if easy:
        return " ".join(easy) if isinstance(easy, list) else easy
    return None


In [ ]:
results = []
N = len(val_records)

for i, r in tqdm(list(enumerate(val_records[:N])), desc="Evaluating"):
    original_text = r["normal_text"]
    reference     = get_reference(r)

    if reference is None:
        continue

    # Baseline simplification
    baseline_prompt = build_baseline_prompt(original_text)
    baseline_output = call_llm(client, baseline_prompt)

    # Graph-guided simplification
    normal_amrs = r["normal_amr"] if isinstance(r["normal_amr"], list) else [r["normal_amr"]]
    doc_G = build_doc_graph(normal_amrs)
    _, _, ranked = extract_knowledge_graph(original_text, nlp)
    graph_prompt = build_llm_prompt(original_text, doc_G, ranked)
    graph_guided_output = call_llm(client, graph_prompt)

    if baseline_output is None or graph_guided_output is None:
        continue

    results.append({
        "id":                  r.get("id", str(i)),
        "original_text":       original_text,
        "reference_easy_text": reference,
        "baseline_output":     baseline_output,
        "graph_guided_output": graph_guided_output,
        "top_concepts":        ranked[:10],
        "graph_prompt":        graph_prompt,
    })
    time.sleep(1)

print(f"\nCompleted {len(results)}/{N} records")

LOCAL_PATH = "results/llm_all_data_full_results.jsonl"
with open(LOCAL_PATH, "w") as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
print("Saved:", LOCAL_PATH)


In [ ]:
"""
Evaluation of Performance

Measures baseline vs graph guided simplfiication on four metrics:
1. SARI: primary simplification quality metric
2. BLEU: n gram overlap with reference
3. Flesch-Kincaid: readability/grade level
4. Concept preservation
"""

# Imports
!pip install -q syllables nltk git+https://github.com/feralvam/easse.git --break-system-packages
import json, re
import syllables
from easse.sari import corpus_sari
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Load results
RESULTS_PATH = "results/llm_all_data_full_results.jsonl"

records = []
with open(RESULTS_PATH) as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print(f"Loaded {len(records)} records for evaluation")


# Metric 1: SARI

def compute_sari(original, hypothesis, reference):
    return round(corpus_sari([original], [hypothesis], [[reference]]), 2)

# Metric 2: BLEU

def compute_bleu(hypothesis, reference):
    hyp  = hypothesis.lower().split()
    ref  = reference.lower().split()
    sf   = SmoothingFunction().method1
    return round(sentence_bleu([ref], hyp, smoothing_function=sf), 4)

# Metric 3: Flesch-Kincaid

def flesch_kincaid_grade(text):
    sents = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    words = [w for w in text.split() if w.strip()]
    if not sents or not words:
        return 0.0
    syl_count = sum(syllables.estimate(w) for w in words)
    asl = len(words) / len(sents)        # average sentence length
    asw = syl_count / len(words)         # average syllables per word
    return round(0.39 * asl + 11.8 * asw - 15.59, 2)


# Metric 4: Concept preservation

def concept_preservation(output_text, top_concepts, top_n=5):
    out_lower = output_text.lower()
    must = top_concepts[:top_n]
    if not must:
        return 0.0
    found = sum(1 for concept, _ in must if concept.lower() in out_lower)
    return round(found / len(must), 3)


# Average sentence length
# Not necessarily a performance metric, but helpful for contextualizing complexity
def avg_sentence_length(text):
    sents = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    if not sents:
        return 0.0
    return round(sum(len(s.split()) for s in sents) / len(sents), 1)

def clean(text):
    return text.replace("  \n", " ").replace("\n", " ").strip()

rows = []
for r in records:
    orig     = r["original_text"]
    ref      = r["reference_easy_text"]
    base     = clean(r["baseline_output"])
    graph    = clean(r["graph_guided_output"])
    concepts = r["top_concepts"]

    rows.append({
        "id":         r["id"],
        "sari_base":  compute_sari(orig, base,  ref),
        "sari_graph": compute_sari(orig, graph, ref),
        "bleu_base":  compute_bleu(base,  ref),
        "bleu_graph": compute_bleu(graph, ref),
        "fk_orig":    flesch_kincaid_grade(orig),
        "fk_ref":     flesch_kincaid_grade(ref),
        "fk_base":    flesch_kincaid_grade(base),
        "fk_graph":   flesch_kincaid_grade(graph),
        "cp_base":    concept_preservation(base,  concepts),
        "cp_graph":   concept_preservation(graph, concepts),
        "asl_base":   avg_sentence_length(base),
        "asl_graph":  avg_sentence_length(graph),
    })

print(f"\n{'ID':<8} {'SARI_B':>7} {'SARI_G':>7} {'ΔSARI':>7}  "
      f"{'BLEU_B':>7} {'BLEU_G':>7}  "
      f"{'FK_B':>5} {'FK_G':>5} {'FK_R':>5}  "
      f"{'CP_B':>5} {'CP_G':>5}")
print("─" * 92)

for row in rows:
    delta = round(row["sari_graph"] - row["sari_base"], 2)
    sign  = "+" if delta >= 0 else ""
    print(
        f"{row['id']:<8} {row['sari_base']:>7} {row['sari_graph']:>7} {sign}{delta:>6.2f}  "
        f"{row['bleu_base']:>7.4f} {row['bleu_graph']:>7.4f}  "
        f"{row['fk_base']:>5} {row['fk_graph']:>5} {row['fk_ref']:>5}  "
        f"{row['cp_base']:>5} {row['cp_graph']:>5}"
    )


def mean(vals):
    return round(sum(vals) / len(vals), 3)

sari_b  = mean([r["sari_base"]  for r in rows])
sari_g  = mean([r["sari_graph"] for r in rows])
bleu_b  = mean([r["bleu_base"]  for r in rows])
bleu_g  = mean([r["bleu_graph"] for r in rows])
fk_orig = mean([r["fk_orig"]    for r in rows])
fk_b    = mean([r["fk_base"]    for r in rows])
fk_g    = mean([r["fk_graph"]   for r in rows])
fk_ref  = mean([r["fk_ref"]     for r in rows])
cp_b    = mean([r["cp_base"]    for r in rows])
cp_g    = mean([r["cp_graph"]   for r in rows])
asl_b   = mean([r["asl_base"]   for r in rows])
asl_g   = mean([r["asl_graph"]  for r in rows])

sari_delta = round(sari_g - sari_b, 3)

# save scores as JSON
SCORES_PATH = "results/evaluation_scores_all.json"

output = {
    "n_records": len(rows),
    "corpus": {
        "sari": {
            "baseline": sari_b,
            "graph": sari_g,
            "delta": sari_delta
        },
        "bleu": {
            "baseline": bleu_b,
            "graph": bleu_g
        },
        "flesch_kincaid_grade": {
            "original": fk_orig,
            "baseline": fk_b,
            "graph": fk_g,
            "reference": fk_ref
        },
        "concept_preservation": {
            "baseline": cp_b,
            "graph": cp_g
        },
        "average_sentence_length": {
            "baseline": asl_b,
            "graph": asl_g
        }
    },
    "per_record": rows
}

with open(SCORES_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

# print(f"saved evaluation JSON to {SCORES_PATH}")

In [ ]:
# Plot results

import json
import matplotlib.pyplot as plt
import numpy as np
import os

SCORES_PATH = "results/evaluation_scores_all.json"
PLOTS_DIR = "results/plots_all"
os.makedirs(PLOTS_DIR, exist_ok=True)

with open(SCORES_PATH, "r", encoding="utf-8") as f:
    scores = json.load(f)

rows = scores["per_record"]
corpus = scores["corpus"]

ids = [str(r["id"]) for r in rows]
x = np.arange(len(ids))
width = 0.38


In [ ]:
metrics = ["SARI", "BLEU", "Concept preservation", "Avg sentence length"]
baseline = [
    corpus["sari"]["baseline"],
    corpus["bleu"]["baseline"],
    corpus["concept_preservation"]["baseline"],
    corpus["average_sentence_length"]["baseline"],
]
graph = [
    corpus["sari"]["graph"],
    corpus["bleu"]["graph"],
    corpus["concept_preservation"]["graph"],
    corpus["average_sentence_length"]["graph"],
]

x_metrics = np.arange(len(metrics))

plt.figure(figsize=(10, 5))
plt.bar(x_metrics - width/2, baseline, width, label="Baseline")
plt.bar(x_metrics + width/2, graph, width, label="Graph-guided")
plt.xticks(x_metrics, metrics, rotation=20, ha="right")

plt.ylabel("Score")
plt.title("Corpus-Level Evaluation: Baseline vs Graph-Guided")
plt.legend()
plt.tight_layout()

plt.savefig(f"{PLOTS_DIR}/corpus_metric_comparison.png", dpi=300)
plt.show()

In [ ]:
sari_base = [r["sari_base"] for r in rows]
sari_graph = [r["sari_graph"] for r in rows]
ids = [r["id"] for r in rows]

x = np.arange(len(rows))
width = 0.35

wins = sum(r["sari_graph"] > r["sari_base"] for r in rows)
losses = sum(r["sari_graph"] < r["sari_base"] for r in rows)
ties = sum(r["sari_graph"] == r["sari_base"] for r in rows)

print("Graph-guided wins:", wins)
print("Baseline wins:", losses)
print("Ties:", ties)
print("Win rate:", round(wins / len(rows), 3))

plt.figure(figsize=(12, 5))
plt.bar(x - width/2, sari_base, width, label="Baseline")
plt.bar(x + width/2, sari_graph, width, label="Graph-guided")

plt.ylabel("SARI")
plt.title("Per-Record SARI Scores")
plt.legend()
plt.tight_layout()

plt.savefig(f"{PLOTS_DIR}/per_record_sari.png", dpi=300)
plt.show()

In [ ]:
sari_delta = [r["sari_graph"] - r["sari_base"] for r in rows]

plt.figure(figsize=(12, 5))
plt.bar(ids, sari_delta)
plt.axhline(0, linewidth=1)
# plt.xticks(rotation=45, ha="right")
plt.xlabel("Records")
plt.xticks([])
plt.ylabel("Graph SARI - Baseline SARI")
plt.title("SARI Improvement from Graph Guidance")
plt.tight_layout()

plt.savefig(f"{PLOTS_DIR}/sari_improvement.png", dpi=300)
plt.show()

In [ ]:
fk_orig = corpus["flesch_kincaid_grade"]["original"]
fk_base = corpus["flesch_kincaid_grade"]["baseline"]
fk_graph = corpus["flesch_kincaid_grade"]["graph"]
fk_ref = corpus["flesch_kincaid_grade"]["reference"]

labels = ["Original", "Baseline", "Graph-guided", "Reference"]
values = [fk_orig, fk_base, fk_graph, fk_ref]

plt.figure(figsize=(8, 5))
plt.bar(labels, values)
plt.ylabel("Flesch-Kincaid Grade Level")
plt.title("Readability Comparison")
plt.tight_layout()

plt.savefig(f"{PLOTS_DIR}/fk_readability_comparison.png", dpi=300)
plt.show()

In [ ]:
cp_base = [r["cp_base"] for r in rows]
cp_graph = [r["cp_graph"] for r in rows]

plt.figure(figsize=(12, 5))
plt.bar(x - width/2, cp_base, width, label="Baseline")
plt.bar(x + width/2, cp_graph, width, label="Graph-guided")
# plt.xticks(x, ids, rotation=45, ha="right")
plt.xlabel("Records")
plt.xlabel([])
plt.ylabel("Concept Preservation Rate")
plt.title("Per-Record Concept Preservation")
plt.legend()
plt.tight_layout()

plt.savefig(f"{PLOTS_DIR}/concept_preservation.png", dpi=300)
plt.show()

In [ ]:
"""
Statistics computation
Here, we'll see the performance of the graph guided prompt against the baseline
prompt performance.
"""

import numpy as np
from scipy import stats as scipy_stats
import json

# helper functions
def arr(key):
    return np.array([r[key] for r in rows], dtype=float)

def fmt(val, decimals=3):
    return round(float(val), decimals)

def paired_ttest(a, b, label):
    """Two tailed paired t-test; prints t, p, as well as significance."""
    t, p = scipy_stats.ttest_rel(a, b)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"  {label:<35} t={t:+.3f}  p={p:.4f}  {sig}")
    return t, p

def wilcoxon(a, b, label):
    """Wilcoxon signed-rank test (t-test alternative)"""
    try:
        stat, p = scipy_stats.wilcoxon(a - b)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        print(f"  {label:<35} W={stat:.1f}   p={p:.4f}  {sig}")
        return stat, p
    except Exception as e:
        print(f"  {label:<35} wilcoxon failed: {e}")
        return None, None

def describe(arr_vals, label):
    """Print mean and standard deviation, median, min, and max."""
    print(f"  {label}")
    print(f"    mean ± SD : {fmt(np.mean(arr_vals))} ± {fmt(np.std(arr_vals))}")
    print(f"    median    : {fmt(np.median(arr_vals))}")
    print(f"    min / max : {fmt(np.min(arr_vals))} / {fmt(np.max(arr_vals))}")

# stats arrays
sari_b_arr  = arr("sari_base")
sari_g_arr  = arr("sari_graph")
bleu_b_arr  = arr("bleu_base")
bleu_g_arr  = arr("bleu_graph")
fk_orig_arr = arr("fk_orig")
fk_ref_arr  = arr("fk_ref")
fk_b_arr    = arr("fk_base")
fk_g_arr    = arr("fk_graph")
cp_b_arr    = arr("cp_base")
cp_g_arr    = arr("cp_graph")
asl_b_arr   = arr("asl_base")
asl_g_arr   = arr("asl_graph")

sari_delta_arr = sari_g_arr - sari_b_arr
bleu_delta_arr = bleu_g_arr - bleu_b_arr
cp_delta_arr   = cp_g_arr   - cp_b_arr
fk_delta_arr   = fk_g_arr   - fk_b_arr

N = len(rows)

# corpus level means
print("=" * 65)
print("1. CORPUS-LEVEL MEANS")
print("=" * 65)
print(f"  N records evaluated : {N}")
print()
print(f"  {'Metric':<30} {'Baseline':>10} {'Graph':>10} {'Δ':>10}")
print(f"  {'-'*30} {'-'*10} {'-'*10} {'-'*10}")
print(f"  {'SARI':<30} {fmt(np.mean(sari_b_arr)):>10} {fmt(np.mean(sari_g_arr)):>10} {fmt(np.mean(sari_delta_arr)):>+10}")
print(f"  {'BLEU':<30} {fmt(np.mean(bleu_b_arr)):>10} {fmt(np.mean(bleu_g_arr)):>10} {fmt(np.mean(bleu_delta_arr)):>+10}")
print(f"  {'FK Grade (output)':<30} {fmt(np.mean(fk_b_arr)):>10} {fmt(np.mean(fk_g_arr)):>10} {fmt(np.mean(fk_delta_arr)):>+10}")
print(f"  {'FK Grade (original)':<30} {fmt(np.mean(fk_orig_arr)):>10} {'—':>10} {'—':>10}")
print(f"  {'FK Grade (reference)':<30} {'—':>10} {fmt(np.mean(fk_ref_arr)):>10} {'—':>10}")
print(f"  {'Concept Preservation':<30} {fmt(np.mean(cp_b_arr)):>10} {fmt(np.mean(cp_g_arr)):>10} {fmt(np.mean(cp_delta_arr)):>+10}")
print(f"  {'Avg Sentence Length':<30} {fmt(np.mean(asl_b_arr)):>10} {fmt(np.mean(asl_g_arr)):>10} {fmt(np.mean(asl_g_arr - asl_b_arr)):>+10}")

# standard deviations
print()
print("=" * 65)
print("2. STANDARD DEVIATIONS  (mean ± SD)")
print("=" * 65)
for label, b_arr, g_arr in [
    ("SARI",                 sari_b_arr,  sari_g_arr),
    ("BLEU",                 bleu_b_arr,  bleu_g_arr),
    ("FK Grade (output)",    fk_b_arr,    fk_g_arr),
    ("Concept Preservation", cp_b_arr,    cp_g_arr),
    ("Avg Sentence Length",  asl_b_arr,   asl_g_arr),
]:
    print(f"  {label}")
    print(f"    Baseline : {fmt(np.mean(b_arr))} ± {fmt(np.std(b_arr))}")
    print(f"    Graph    : {fmt(np.mean(g_arr))} ± {fmt(np.std(g_arr))}")

# per record Sari wins, losses, and ties
print()
print("=" * 65)
print("3. PER-RECORD SARI WIN / LOSS / TIE")
print("=" * 65)
wins   = int(np.sum(sari_delta_arr > 0))
losses = int(np.sum(sari_delta_arr < 0))
ties   = int(np.sum(sari_delta_arr == 0))
print(f"  Graph-guided wins  : {wins}  ({fmt(wins/N*100, 1)}%)")
print(f"  Baseline wins      : {losses}  ({fmt(losses/N*100, 1)}%)")
print(f"  Ties               : {ties}  ({fmt(ties/N*100, 1)}%)")
print()
print(f"  SARI delta distribution:")
describe(sari_delta_arr, "Graph SARI − Baseline SARI")
print(f"  Records with Δ > +2 : {int(np.sum(sari_delta_arr > 2))}")
print(f"  Records with Δ < -2 : {int(np.sum(sari_delta_arr < -2))}")

# readability
print()
print("=" * 65)
print("4. READABILITY (FLESCH-KINCAID GRADE LEVEL)")
print("=" * 65)
fk_orig_mean = fmt(np.mean(fk_orig_arr))
fk_ref_mean  = fmt(np.mean(fk_ref_arr))
fk_b_mean    = fmt(np.mean(fk_b_arr))
fk_g_mean    = fmt(np.mean(fk_g_arr))

print(f"  Original text (input)  : {fk_orig_mean}")
print(f"  Human reference        : {fk_ref_mean}")
print(f"  Baseline output        : {fk_b_mean}   (reduction from original: {fmt(fk_b_mean - fk_orig_mean):+})")
print(f"  Graph-guided output    : {fk_g_mean}   (reduction from original: {fmt(fk_g_mean - fk_orig_mean):+})")
print(f"  Graph vs baseline Δ    : {fmt(fk_g_mean - fk_b_mean):+}")
print(f"  Graph vs reference Δ   : {fmt(fk_g_mean - fk_ref_mean):+}")
print()
pct_b_better = fmt(np.mean(fk_b_arr < fk_orig_arr) * 100, 1)
pct_g_better = fmt(np.mean(fk_g_arr < fk_orig_arr) * 100, 1)
print(f"  % records where baseline output < original FK  : {pct_b_better}%")
print(f"  % records where graph output    < original FK  : {pct_g_better}%")

# signficance tests
print()
print("=" * 65)
print("5. SIGNIFICANCE TESTS")
print("   (paired t-test + Wilcoxon signed-rank, graph vs baseline)")
print("   Significance: * p<0.05  ** p<0.01  *** p<0.001  ns=not significant")
print("=" * 65)
print()
print("  Paired t-tests:")
paired_ttest(sari_g_arr,  sari_b_arr,  "SARI (graph > baseline)")
paired_ttest(bleu_g_arr,  bleu_b_arr,  "BLEU (graph > baseline)")
paired_ttest(fk_g_arr,    fk_b_arr,    "FK Grade (graph vs baseline)")
paired_ttest(cp_g_arr,    cp_b_arr,    "Concept Preservation")
print()
print("  Wilcoxon signed-rank tests (non-parametric):")
wilcoxon(sari_g_arr,  sari_b_arr,  "SARI (graph > baseline)")
wilcoxon(bleu_g_arr,  bleu_b_arr,  "BLEU (graph > baseline)")
wilcoxon(fk_g_arr,    fk_b_arr,    "FK Grade (graph vs baseline)")
wilcoxon(cp_g_arr,    cp_b_arr,    "Concept Preservation")

# concept preservation
print()
print("=" * 65)
print("8. CONCEPT PRESERVATION DETAIL")
print("=" * 65)
print(f"  Baseline  mean ± SD : {fmt(np.mean(cp_b_arr))} ± {fmt(np.std(cp_b_arr))}")
print(f"  Graph     mean ± SD : {fmt(np.mean(cp_g_arr))} ± {fmt(np.std(cp_g_arr))}")
print(f"  % records graph ≥ 1.0 (perfect) : {fmt(np.mean(cp_g_arr == 1.0)*100, 1)}%")
print(f"  % records baseline ≥ 1.0        : {fmt(np.mean(cp_b_arr == 1.0)*100, 1)}%")
print(f"  % records graph > baseline      : {fmt(np.mean(cp_g_arr > cp_b_arr)*100, 1)}%")
print(f"  % records graph < baseline      : {fmt(np.mean(cp_g_arr < cp_b_arr)*100, 1)}%")

PAPER_STATS_PATH = "/content/drive/My Drive/COMP459/outputs/paper_statistics.json"

t_sari, p_sari   = scipy_stats.ttest_rel(sari_g_arr,  sari_b_arr)
t_bleu, p_bleu   = scipy_stats.ttest_rel(bleu_g_arr,  bleu_b_arr)
t_fk,   p_fk     = scipy_stats.ttest_rel(fk_g_arr,    fk_b_arr)
t_cp,   p_cp     = scipy_stats.ttest_rel(cp_g_arr,    cp_b_arr)

paper_stats = {
    "n_records": N,
    "corpus_means": {
        "sari":   {"baseline": fmt(np.mean(sari_b_arr)),  "graph": fmt(np.mean(sari_g_arr)),  "delta": fmt(np.mean(sari_delta_arr))},
        "bleu":   {"baseline": fmt(np.mean(bleu_b_arr)),  "graph": fmt(np.mean(bleu_g_arr)),  "delta": fmt(np.mean(bleu_delta_arr))},
        "fk":     {"original": fmt(np.mean(fk_orig_arr)), "reference": fmt(np.mean(fk_ref_arr)),
                   "baseline": fmt(np.mean(fk_b_arr)),    "graph": fmt(np.mean(fk_g_arr)),    "delta": fmt(np.mean(fk_delta_arr))},
        "cp":     {"baseline": fmt(np.mean(cp_b_arr)),    "graph": fmt(np.mean(cp_g_arr)),    "delta": fmt(np.mean(cp_delta_arr))},
        "asl":    {"baseline": fmt(np.mean(asl_b_arr)),   "graph": fmt(np.mean(asl_g_arr))},
    },
    "standard_deviations": {
        "sari":  {"baseline": fmt(np.std(sari_b_arr)),  "graph": fmt(np.std(sari_g_arr))},
        "bleu":  {"baseline": fmt(np.std(bleu_b_arr)),  "graph": fmt(np.std(bleu_g_arr))},
        "fk":    {"baseline": fmt(np.std(fk_b_arr)),    "graph": fmt(np.std(fk_g_arr))},
        "cp":    {"baseline": fmt(np.std(cp_b_arr)),    "graph": fmt(np.std(cp_g_arr))},
        "asl":   {"baseline": fmt(np.std(asl_b_arr)),   "graph": fmt(np.std(asl_g_arr))},
    },
    "win_loss_tie": {
        "wins": wins, "losses": losses, "ties": ties,
        "win_rate": fmt(wins / N),
        "win_pct":  fmt(wins / N * 100, 1),
        "loss_pct": fmt(losses / N * 100, 1),
        "tie_pct":  fmt(ties / N * 100, 1),
    },
    "sari_delta": {
        "mean":   fmt(np.mean(sari_delta_arr)),
        "sd":     fmt(np.std(sari_delta_arr)),
        "median": fmt(np.median(sari_delta_arr)),
        "min":    fmt(np.min(sari_delta_arr)),
        "max":    fmt(np.max(sari_delta_arr)),
        "n_gt2":  int(np.sum(sari_delta_arr > 2)),
        "n_lt_neg2": int(np.sum(sari_delta_arr < -2)),
    },
    "readability": {
        "pct_baseline_below_original": fmt(np.mean(fk_b_arr < fk_orig_arr) * 100, 1),
        "pct_graph_below_original":    fmt(np.mean(fk_g_arr < fk_orig_arr) * 100, 1),
        "graph_vs_reference_delta":    fmt(np.mean(fk_g_arr) - np.mean(fk_ref_arr)),
    },
    "significance_tests": {
        "sari": {"t": fmt(t_sari, 4), "p": fmt(p_sari, 4)},
        "bleu": {"t": fmt(t_bleu, 4), "p": fmt(p_bleu, 4)},
        "fk":   {"t": fmt(t_fk,   4), "p": fmt(p_fk,   4)},
        "cp":   {"t": fmt(t_cp,   4), "p": fmt(p_cp,   4)},
    },
    "concept_preservation_detail": {
        "pct_graph_perfect":       fmt(np.mean(cp_g_arr == 1.0) * 100, 1),
        "pct_baseline_perfect":    fmt(np.mean(cp_b_arr == 1.0) * 100, 1),
        "pct_graph_beats_baseline": fmt(np.mean(cp_g_arr > cp_b_arr) * 100, 1),
        "pct_graph_worse":         fmt(np.mean(cp_g_arr < cp_b_arr) * 100, 1),
    },
}

with open(PAPER_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(paper_stats, f, indent=2, ensure_ascii=False)

print()
print("=" * 65)
print(f"All paper statistics saved to:\n  {PAPER_STATS_PATH}")
print("=" * 65)

In [ ]:
bleu_base  = [r["bleu_base"]  for r in rows]
bleu_graph = [r["bleu_graph"] for r in rows]
bleu_delta = [r["bleu_graph"] - r["bleu_base"] for r in rows]

x     = np.arange(len(rows))
width = 0.35

plt.figure(figsize=(8, 5))
plt.hist(bleu_base,  bins=40, alpha=0.6, label="Baseline",     color="#1f77b4")
plt.hist(bleu_graph, bins=40, alpha=0.6, label="Graph-guided", color="#ff7f0e")
plt.axvline(corpus["bleu"]["baseline"], color="#1f77b4", linestyle="--",
            linewidth=1.5, label=f"Baseline mean ({corpus['bleu']['baseline']:.3f})")
plt.axvline(corpus["bleu"]["graph"],    color="#ff7f0e", linestyle="--",
            linewidth=1.5, label=f"Graph mean ({corpus['bleu']['graph']:.3f})")
plt.xlabel("BLEU Score")
plt.ylabel("Number of Records")
plt.title("BLEU Score Distribution: Baseline vs Graph-Guided")
plt.legend()
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/bleu_distribution.png", dpi=300)
plt.show()